# 01 — Exploration

## Project Question
Which factors among delivery performance, product attributes, seller reliability, 
and order characteristics have the strongest measurable association with 1-star 
reviews on Olist?

## Step 2 — Data Understanding

### Schema Map
| Table | Description | Use in Project |
|---|---|---|
| orders | Each row = one order | YES — analytical spine |
| order_items | Each row = one line item per order | YES — link products + sellers |
| products | Product attributes | YES — product factor |
| sellers | Seller info | YES — seller factor |
| reviews | Customer review per order | YES — dependent variable |
| payments | Order payment info | MAYBE — verify in EDA |
| customers | Customer info (one row per order, not per person) | MAYBE — only for repeat-customer linkage |
| category_translation | Portuguese → English category names | MAYBE — readability |
| geolocation | ZIP code → lat/long lookup | NO — not in factor scope |

### Joining Logic
[Sketch how the tables connect — which keys join to which]

- orders.order_id → order_items.order_id
- orders.order_id → reviews.order_id
- orders_items.seller_id → sellers.seller_id
- orders_items.product_id → products.product_id


In [10]:
import pandas as pd 

# Load core tables
orders = pd.read_csv('../data/olist_orders_dataset.csv')
order_items = pd.read_csv('../data/olist_order_items_dataset.csv')
reviews = pd.read_csv('../data/olist_order_reviews_dataset.csv')
products = pd.read_csv('../data/olist_products_dataset.csv')
sellers = pd.read_csv('../data/olist_sellers_dataset.csv')

tables = {
    'orders': orders,
    'order_items': order_items,
    'reviews': reviews,
    'products': products,
    'sellers': sellers,
}

for name, df in tables.items():
    print(f"\n=== {name} ===")
    print(f"Shape: {df.shape}")
    print(df.dtypes)

    print(f"=== Missing values ===")
    print(df.isna().sum())
    print("\n")
   
    
    print(f"=== DataFrame {name} Head ===")
    print(df.head())
    print("\n")


=== orders ===
Shape: (99441, 8)
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object
=== Missing values ===
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64


=== DataFrame orders Head ===
                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a36

- in orders table have 3 columns(order_approved_at, order_delivered_carrier_date, order_delivered_customer_date ) with missing values with order_delivered_customer_date column having highest number of missing values
- order_status is of type strshould be a category, order_purchase_timestamp, order_approved_at, order_delivered_carrier_date ,order_delivered_customer_date, order_estimated_delivery_date, is of type str should be datetime
- order-items table has no missing values accross all columns
- in order items table shipping_limit_date is of type str but should be datetime
- reviews table has 2 columns with missing values review_comment_title column has 87656 and review_comment_message column has 58247
- in reviews table two columns which should be datetime type are of type str review_creation_date and review_answer_timestamp columns str
- in products table the following ncolumns have missing values               product_category_name 610
    product_name_lenght 610
    product_description_lenght 610
    product_photos_qty 610
    product_weight_g 2
    product_length_cm 2
    product_height_cm 2
    product_width_cm 2
- all columns in products table have correct types
- in sellers table seller_city and seller_state are of type str but sjhould be type category
- sellers table has no missing values

In [4]:
df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 21.9 MB


In [11]:
reviews['review_score'].value_counts().sort_index()
print(f"\n1-star reviews: {(reviews['review_score'] == 1).sum()} ({(reviews['review_score'] == 1).mean()*100:.1f}%)")


1-star reviews: 11424 (11.5%)
